# Data Analysis

In [57]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
import requests
import plotly.express as px
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer
from matplotlib.pyplot import scatter

print("All libraries imported successfully.")

All libraries imported successfully.


In [58]:
# Import the cleaned dataset
merged = pd.read_csv("../data/processed/merged_dataset.csv")
print("Merged dataset imported successfully with shape:", merged.shape)

Merged dataset imported successfully with shape: (152, 17)


## Start by showing a world map of financial account ownership % by country

In [59]:
world_map_analysis = merged[["country", "country_code", "percent_with_fin_account"]].copy()

#Rewrite shares as percentages instead of decimals
world_map_analysis["percent_with_fin_account"] = world_map_analysis["percent_with_fin_account"] * 100

# Create the first choropleth map showing percent with financial account by country
world_map_fig = px.choropleth(
    world_map_analysis,
    locations="country_code",
    color="percent_with_fin_account",
    hover_name="country",
    hover_data={"country_code": False, "percent_with_fin_account": ':.2f'}, # Round % to 2 decimal place in hover tooltip
    color_continuous_scale="Blues",
    range_color=(0, 100),
    projection="robinson",
    labels={"percent_with_fin_account": "Account Ownership (%)"},
    title="Percentage of Adults with a Financial Account by Country"
)

# Clean up the map layout
world_map_fig.update_geos(showframe=False, showcoastlines=False)
world_map_fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))

# Export the map as an interactive HTML file
world_map_fig.write_html("../outputs/account_ownership_map.html")


## Bar chart of regions ranked by financial inclusion

In [60]:
# Define function to create a weighted average
def weighted_avg(df, value_col, weight_col="adult_pop"):
    sub = df.dropna(subset=[value_col, weight_col])
    return (sub[value_col] * sub[weight_col]).sum() / sub[weight_col].sum()

#Calculate average percent with financial account by geographic region
region_df = (
    merged.groupby("region_geo")
    .apply(lambda g: weighted_avg(g, "percent_with_fin_account"),
           include_groups=False)
    .reset_index(name="percent_with_fin_account")
    .sort_values("percent_with_fin_account", ascending=True)
    )

#Create bar chart showing average percent with financial account by geographic region
region_fig = px.bar(
    region_df,
    x="percent_with_fin_account",
    y="region_geo",
    orientation="h",
    color="percent_with_fin_account",
    color_continuous_scale="Blues",
    title="Financial Inclusion by Geographic Region (avg % of adults with a financial account)",
    labels={"percent_with_fin_account": "Average Account Ownership (%)", "region_geo": "Region"}
)

region_fig.update_traces(textposition="outside", hovertemplate="<b>%{y}</b><br>Average Account Ownership: %{x:.2%}<extra></extra>")
region_fig.update_layout(coloraxis_showscale=False, xaxis_tickformat=".0%")
region_fig.write_html("../outputs/account_ownership_by_region.html")


## Bar chart: unbanked by gender, income, education

In [61]:
#Check summary statistics for the demographic variables
demo_cols = ["account_female", "account_male", "account_poor40", 
             "account_rich60", "account_prim_less", "account_sec_more"]
print(merged[demo_cols].describe())

       account_female  account_male  account_poor40  account_rich60  \
count      147.000000    147.000000      146.000000      146.000000   
mean        67.412743     73.581184       63.440696       75.200739   
std         24.702032     20.600126       25.463110       21.057782   
min         11.912170     17.613956       13.606254       13.052727   
25%         46.947277     59.462067       42.663412       60.739351   
50%         71.489046     75.566924       66.701547       81.104868   
75%         88.132886     92.208543       85.310752       93.844898   
max         99.848495    100.000000      100.000000       99.863252   

       account_prim_less  account_sec_more  
count         147.000000        147.000000  
mean           59.465558         77.834687  
std            24.687694         18.343444  
min            13.270897         26.277648  
25%            40.508842         65.912015  
50%            57.452222         80.888848  
75%            80.796483         94.034214  


In [62]:
# Divide by 100 to convert to decimals to match the percent_with_fin_account variable
for col in demo_cols:
    merged[col] = merged[col] / 100     

print(merged[demo_cols].describe())

       account_female  account_male  account_poor40  account_rich60  \
count      147.000000    147.000000      146.000000      146.000000   
mean         0.674127      0.735812        0.634407        0.752007   
std          0.247020      0.206001        0.254631        0.210578   
min          0.119122      0.176140        0.136063        0.130527   
25%          0.469473      0.594621        0.426634        0.607394   
50%          0.714890      0.755669        0.667015        0.811049   
75%          0.881329      0.922085        0.853108        0.938449   
max          0.998485      1.000000        1.000000        0.998633   

       account_prim_less  account_sec_more  
count         147.000000        147.000000  
mean            0.594656          0.778347  
std             0.246877          0.183434  
min             0.132709          0.262776  
25%             0.405088          0.659120  
50%             0.574522          0.808888  
75%             0.807965          0.940342  


In [63]:
# Compute population-weighted average percent with financial account for each demographic group

def weighted_unbanked(df, account_col, weight_col="adult_pop"):
    """Population-weighted unbanked rate (1 - account ownership)."""
    sub = df.dropna(subset=[account_col, weight_col])
    weighted_account = (sub[account_col] * sub[weight_col]).sum() / sub[weight_col].sum()
    return 1 - weighted_account

# Create a DataFrame to summarize the unbanked rates by demographic group

rows = [
    ("Gender",    "Women",              weighted_unbanked(merged, "account_female")),
    ("Gender",    "Men",                weighted_unbanked(merged, "account_male")),
    ("Income",    "Poorest 40%",        weighted_unbanked(merged, "account_poor40")),
    ("Income",    "Richest 60%",        weighted_unbanked(merged, "account_rich60")),
    ("Education", "Primary or less",    weighted_unbanked(merged, "account_prim_less")),
    ("Education", "Secondary or more",  weighted_unbanked(merged, "account_sec_more")),
]

unbanked_df = pd.DataFrame(rows, columns=["dimension", "group", "unbanked_rate"])
print(unbanked_df)

   dimension              group  unbanked_rate
0     Gender              Women       0.241613
1     Gender                Men       0.198771
2     Income        Poorest 40%       0.284384
3     Income        Richest 60%       0.177846
4  Education    Primary or less       0.303822
5  Education  Secondary or more       0.156998


In [64]:
# Label groups as "Advantaged" vs "Disadvantaged" for coloring the bar chart
unbanked_df["Status"] = unbanked_df["group"].map({
    "Women": "Disadvantaged group",
    "Men": "Advantaged group",
    "Poorest 40%": "Disadvantaged group",
    "Richest 60%": "Advantaged group",
    "Primary or less": "Disadvantaged group",
    "Secondary or more": "Advantaged group",
})

# Create bar chart showing unbanked rates by demographic group

unbanked_fig = px.bar(
    unbanked_df,
    x="dimension",
    y="unbanked_rate",
    color="Status",
    barmode="group",
    hover_data=["group"],
    text=unbanked_df["unbanked_rate"].mul(100).round(1).astype(str) + "%",
    color_discrete_map={"Advantaged group": "#CADFF0", "Disadvantaged group": "#1360A8"},
    title="Unbanked rates by gender, income, and education",
    labels={"unbanked_rate": "% unbanked", "dimension": ""},
)
unbanked_fig.update_traces(textposition="outside", hovertemplate="<b>%{customdata[0]}</b><br>Account Ownership: %{y:.1%}<extra></extra>")
unbanked_fig.update_layout(yaxis_tickformat=".0%", height=500, plot_bgcolor="white")


unbanked_fig.write_html("../outputs/unbanked_demographic_chart.html")



## Scatter plot: GDP per capita vs. account ownership

In [65]:
# Drop rows missing either variable for the scatter

scatter_data = merged.dropna(subset=["gdp_per_capita", "percent_with_fin_account"])

# Create cool palette for the scatter points, with darker blue for higher GDP per capita to match blog theme while keeping distinguishability
cool_palette = ["#1f4e79",  "#2e8b8b", "#5b8db8", "#7b68a6", "#6cb3c8", "#6b9e6b", "#0a3d62",]


# Create scatter plot showing relationship between GDP per capita and percent with financial account, colored by geographic region and sized by adult population

scatter.fig = px.scatter(
    scatter_data,
    x="gdp_per_capita",
    y="percent_with_fin_account",
    color="region_geo",              
    color_discrete_sequence=cool_palette, 
    size="adult_pop",                     
    size_max=60,
    hover_name="country",
    hover_data={
        "gdp_per_capita": ":$,.0f",
        "percent_with_fin_account": ":.0%",
        "adult_pop": ":,.0f",
    },
    log_x=True,                           
    title="Wealthier countries are more financially included",
    labels={
        "gdp_per_capita": "GDP per capita (log scale, USD)",
        "percent_with_fin_account": "Account ownership",
        "region_geo": "Region",
        "adult_pop": "Adult population"
    },
    trendline="ols",                 
    trendline_scope="overall",            
    trendline_color_override="black",
)

# Make the points a bit nicer with white borders, and clean up the layout

scatter.fig.update_traces(marker=dict(line=dict(width=0.5, color="white")))
scatter.fig.update_layout(
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1],
    height=600,
    plot_bgcolor="white",
    legend=dict(title="", yanchor="bottom", y=0.02, xanchor="right", x=0.98),
)
scatter.fig.update_xaxes(
    tickvals = [500, 1000, 2000, 5000, 10000, 20000, 50000, 100000],
    ticktext=["500", "1k", "2k", "5k", "10k", "20k", "50k", "100k"],
)

scatter.fig.write_html("../outputs/gdp_vs_account_ownership_scatter.html")

In [66]:
# Show the R^2 of the trendline fit to quantify how much of the variation in financial inclusion can be explained by GDP per capita

X = sm.add_constant(np.log(scatter_data["gdp_per_capita"]))
y = scatter_data["percent_with_fin_account"]
r2 = sm.OLS(y, X).fit().rsquared

print(r2)

0.5894283294665991


## Bar Chart: Mobile money adoption vs. traditional banking by region

In [67]:
# Population-weighted averages by region for both account types

regional = (
    merged.groupby("region_geo")
    .apply(lambda g: pd.Series({
        "Traditional banking": weighted_avg(g, "percent_with_fin_account_formal"),
        "Mobile money":        weighted_avg(g, "percent_with_mobile_account"),
    }))
    .reset_index()
    .melt(id_vars="region_geo", var_name="Account type", value_name="share")
)

# Sort regions by traditional banking ascending, so the contrast pops visually
region_order = (
    regional[regional["Account type"] == "Traditional banking"]
    .sort_values("share")["region_geo"].tolist()
)

mobile_fig = px.bar(
    regional,
    x="share",
    y="region_geo",
    color="Account type",
    barmode="group",
    orientation="h",
    category_orders={"region_geo": region_order},
    color_discrete_map={
        "Traditional banking": "#CADFF0",
        "Mobile money": "#1360A8",
    },
    labels={"share": "Share of adults", "region_geo": "Region", "Account type": "Account type"},
)
mobile_fig.update_traces(textposition="outside")
mobile_fig.update_layout(
    xaxis_tickformat=".0%",
    height=500,
    plot_bgcolor="white",
)

mobile_fig.write_html("../outputs/mobile_vs_traditional_by_region.html")

/var/folders/rn/1ww2hhjn0r58pz6lrgp_90sw0000gn/T/ipykernel_16302/3595556314.py:4: RuntimeWarning:

invalid value encountered in scalar divide

/var/folders/rn/1ww2hhjn0r58pz6lrgp_90sw0000gn/T/ipykernel_16302/2531116502.py:5: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



## Regression output table: What predicts inclusion?

In [68]:
reg_data = merged.dropna(subset=[
    "percent_with_fin_account",
    "gdp_per_capita",
    "region_geo",
]).copy()

# Log-transform the variables that span orders of magnitude
reg_data["log_gdp_pc"]  = np.log(reg_data["gdp_per_capita"])

print(f"Sample size: {len(reg_data)} countries")

Sample size: 138 countries


In [69]:
import statsmodels.formula.api as smf

model1 = smf.ols(
    "percent_with_fin_account ~ log_gdp_pc",
    data=reg_data
).fit(cov_type="HC3")

print(model1.summary())

                               OLS Regression Results                               
Dep. Variable:     percent_with_fin_account   R-squared:                       0.589
Model:                                  OLS   Adj. R-squared:                  0.586
Method:                       Least Squares   F-statistic:                     263.9
Date:                      Fri, 01 May 2026   Prob (F-statistic):           1.18e-33
Time:                              12:02:40   Log-Likelihood:                 70.840
No. Observations:                       138   AIC:                            -137.7
Df Residuals:                           136   BIC:                            -131.8
Df Model:                                 1                                         
Covariance Type:                        HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------

In [70]:
# Specification 1: GDP only (the bivariate relationship)
m1 = smf.ols("percent_with_fin_account ~ log_gdp_pc",
             data=reg_data).fit(cov_type="HC3")

# Specification 3: + region fixed effects
m2 = smf.ols("percent_with_fin_account ~ log_gdp_pc + C(region_geo)",
             data=reg_data).fit(cov_type="HC3")

# Specification 4: + mobile money
m3 = smf.ols("percent_with_fin_account ~ log_gdp_pc + C(region_geo) + percent_with_mobile_account",
             data=reg_data.dropna(subset=["percent_with_mobile_account"])).fit(cov_type="HC3")

# Build the side-by-side table
table = Stargazer([m1, m2, m3])
table.title("What predicts financial inclusion?")
table.custom_columns(
    ["GDP only","+ Region FE", "+ Mobile money"],
    [1, 1, 1]
)
table.show_model_numbers(False)
table.rename_covariates({
    "log_gdp_pc": "Log GDP per capita",
    "percent_with_mobile_account": "Mobile money adoption",
})
table.covariate_order(["log_gdp_pc", "percent_with_mobile_account"])  # hides regional dummies
table.add_line("Region fixed effects", ["No", "Yes", "Yes"])
table.show_degrees_of_freedom(False)

with open("../outputs/regression_table.html", "w") as f:
    f.write(table.render_html())
